# BDII -- Sesión 1 -- Procesado inicial de datos

Esta hoja muestra cómo procesar o curar un conjunto de datos para hacerlos más accesibles a la hora de introducirlos en bases de datos. Utilizaremos un conjunto de datos existente en Internet, que se descargará, se procesará y se convertirá en un formato universal como CSV o JSON. En particular se trabajará:

- La descarga de los datos.
- Inspección, identificación del formato y posible procesado.
- Generación de un formato fácilmente digerible por las BBDD, como CSV o JSON.

Comenzaremos instalando los paquetes necesarios:

In [ ]:
!sudo apt-get update -qq

In [ ]:
!sudo apt-get install -y p7zip tree

Importamos algunos paquetes estándar para la hoja

In [ ]:
%pip install pandas matplotlib

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

%matplotlib inline
matplotlib.style.use('ggplot')

In [5]:
RunningInCOLAB = 'google.colab' in str(get_ipython()) if hasattr(__builtins__,'__IPYTHON__') else False

## Datos de Stackoverflow

El conjunto de datos de Stackoverflow es un *dump* de datos que cada cierto tiempo realiza el sitio web stackoverflow.com, en particular, la version en español, http://es.stackoverflow.com. El formato de los datos es XML, aunque es muy sencillo de extraer los datos, como veremos a continuación.

El contenido original se puede descargar directamente de los diferentes _dumps_ que se realizan de la página de archive.org: https://archive.org/details/stackexchange.

Sin embargo, nosotros descargaremos una versión fija previamente descargada para que todos partamos de los mismos datos.

## Descarga de los datos

En este caso los datos están disponibles en un repositorio git. Se pueden descargar también de la Web, pero se van actualizando. Los descargamos del repositorio git para que todos tengáis los mismos.

In [ ]:
!wget https://github.com/dsevilla/bd2-data/raw/24-25/es.stackoverflow/es.stackoverflow.7z.001
!wget https://github.com/dsevilla/bd2-data/raw/24-25/es.stackoverflow/es.stackoverflow.7z.002
!wget https://github.com/dsevilla/bd2-data/raw/24-25/es.stackoverflow/es.stackoverflow.7z.003

In [ ]:
!ls -lh es.stackoverflow.7z*

In [ ]:
!7zr x es.stackoverflow.7z.001

In [ ]:
!ls -lh *.xml

## Inspección y procesado

Podemos inspeccionar los ficheros `.xml` para ver su contenido. Son XML, sí, pero ¿con qué formato?

In [ ]:
!head Posts.xml

Aunque se puede procesar el formato XML, lo que podemos ver es que cada entrada es exactamente una línea que comienza por "`<row`", y que contiene un conjunto de atributos en formato "`atributo="valor"`". Si lo comprobamos, incluso no existirá ninguna comilla doble **dentro** de otra comilla doble, así que podemos extraer esos pares de forma facil.

La siguiente función procesa el fichero XML línea a línea. Primero separa la parte inicial "`<row`", y después procesa cada par clave/valor. Lo único que hace es construir el conjunto de atributos que hay en todas las entradas. Como vimos, cada fila contenía atributos diferentes. Queremos obtenerlos todos.

La función, en vez de retornar una lista, que ocuparía mucha memoria, retorna un generador, que es una lista (de pares clave-valor, un diccionario) que se va generando a medida que se recorre. Por eso utiliza la construcción `yield` de Python. Esto hace que la función se detenga, y cuando se le pide el siguiente elemento, continúa desde donde se quedó (corrutina).

In [11]:
import re
from collections.abc import Iterator

def generate_elements_from_lines(filename: str) -> Iterator[dict[str, str]]:

  def get_attrs(line: str) -> dict[str, str]:
    (_, attrs) = line.split("<row ", 2)
    return {m.group(1): m.group(2)
              for m in re.finditer(r"(\w*?)=\"(.*?)\"", attrs)}

  with open(filename, "r") as f:
    for line in f:
      if "<row" in line:
        yield get_attrs(line)

In [12]:
first_row: dict[str, str] = next(generate_elements_from_lines("Posts.xml"))

In [ ]:
first_row.keys()

Hay que extraer el conjunto de atributos para saber qué columnas tendrá nuestra tabla/CSV o archivo JSON. Recuérdese que las dos primeras filas del archivo XML tenían diferentes atributos. ¿Cómo se haría esto?

In [14]:
from collections.abc import Iterator

def get_all_attrs(iterator: Iterator[dict[str,str]]) -> set[str]:
  all_attrs: set[str] = set()
  for row in iterator:
    all_attrs.update(row.keys())
  return all_attrs

all_attrs: set[str] = get_all_attrs(generate_elements_from_lines("Posts.xml"))

El conjunto de atributos es pues:

In [ ]:
all_attrs

Como sabemos que el atributo `Id` va a ser la clave primaria, lo ponemos al principio. Además, generamos una lista, no un conjunto, para que el orden sea conocido.

In [ ]:
all_attrs.remove('Id')
all_attrs = list(all_attrs)
all_attrs.insert(0,'Id')
all_attrs

## Escritura del formato CSV

El formato CSV está especificado en el estándar RFC 4180. https://www.ietf.org/rfc/rfc4180.txt. En general se puede utilizar la biblioteca `csv` de Python 3 y vamos a exportar una línea de cabecera con todos los campos. https://docs.python.org/3/library/csv.html.

Tendremos en cuenta que todas las filas tienen que tener las mismas columnas y en el mismo orden dado por `all_attrs`.

In [17]:
import csv

def write_csv(destfile: str, all_attrs: list[str], iterator: Iterator[dict[str,str]]) -> None:
  with open(destfile, 'w') as wf:
    cw = csv.writer(wf)

    # Escribir la línea de cabecera
    cw.writerow(all_attrs)

    # Recorrer el iterador
    for row in iterator:
      row_to_write: list[str] = [row.get(att, '') for att in all_attrs]
      cw.writerow(row_to_write)

In [18]:
write_csv('Posts.csv', all_attrs, generate_elements_from_lines('Posts.xml'))

In [ ]:
!head Posts.csv

## Conversión hacia JSON

El siguiente código convierte el fichero CSV en al formato JSON que podéis ver en: https://www.json.org/json-en.html. El código funciona, pero tiene el problema de que para convertir todo a JSON, se tiene que generar un objeto (diccionario) JSON con **todos** los datos, que se tiene que almacenar en memoria. Esto no es siempre posible. Después veremos otro formato que no tiene este problema.

In [20]:
import json

def csv_to_json(fname_csv: str, fname_json: str, primary_key: str) -> None:
    data_dict: dict[str,dict] = {}

    with open(fname_csv, "r") as f_csv:
        csv_reader = csv.DictReader(f_csv)

        for row in csv_reader:
            key: str = row[primary_key]
            data_dict[key] = row

    with open(fname_json, 'w') as f_json:
        f_json.write(json.dumps(data_dict, indent=4))

In [21]:
fname_csv = 'Posts.csv'
fname_json = 'Posts.json'

csv_to_json(fname_csv, fname_json, 'Id')

In [ ]:
!head Posts.json

Si nos damos cuenta, tenemos el problema de que el valor Id está por duplicado.

Vamos a ver cómo eliminar columnas que no queramos tener.


In [23]:
def csv_to_json2(fname_csv: str, fname_json: str, primary_key: str) -> None:
    data_dict: dict[str, dict] = {}

    with open(fname_csv, "r") as f_csv:
        csv_reader = csv.DictReader(f_csv)

        for rows in csv_reader:
            key: str = rows[primary_key]

            # Borramos los campos que nos interesen.
            del rows[primary_key]

            data_dict[key] = rows

    with open(fname_json, 'w') as f_json:
        f_json.write(json.dumps(data_dict, indent=4))

In [24]:
fname_csv = 'Posts.csv'
fname_json = 'Posts.json'

csv_to_json2(fname_csv, fname_json, 'Id')

In [ ]:
!head -n 100 Posts.json

Al escribir en formato JSON se nos queda un fichero compacto que no podemos dividir.

## JSON Lines

Para evitar el problema anterior (que todo el fichero es un JSON gigante que hay que leer en memoria antes de procesarlo), se creó el formato JSON Lines. En vez de tener un array que incluya a todo el fichero, se obliga a que cada objeto JSON incluido en el fichero esté en su propia línea.

Si algún elemento del JSON contiene un salto de línea, se codifica de alguna forma, como por ejemplo como `'\n'`. De esta forma ya están los datos en CSV, así que la conversión no será problemática.

Más información: https://jsonlines.org.

In [26]:
import json

def csv_to_jsonl(fname_csv, fname_jsonl):
    with open(fname_csv, 'r') as f_csv:
        csv_reader = csv.DictReader(f_csv)

        with open(fname_jsonl, 'w') as f_jsonl:
            for row in csv_reader:
                json_line: str = json.dumps(row)
                f_jsonl.write(json_line)
                f_jsonl.write("\n")

In [27]:
csv_to_jsonl('Posts.csv', 'Posts.jsonl')

In [ ]:
!head Posts.jsonl

## Uso de Parquet

![Parquet](https://upload.wikimedia.org/wikipedia/commons/4/47/Apache_Parquet_logo.svg)


El formato Parquet (https://parquet.apache.org) se ha popularizado recientemente con el uso de fuentes de datos en Internet. En general supone una mejora en todos los aspectos con respecto a CSV y en otros con respecto a JSON y JSON lines.

En general, Parquet es un formato de almacenamiento de datos de columnas, que es muy eficiente en términos de espacio y tiempo de acceso. Es un formato binario, pero que se puede leer en muchos lenguajes de programación. Además, permite compresión de datos, lo que lo hace eficiente en tiempo y en espacio.

El formato interno del fichero se describe por encima en la siguiente imagen:

![Parquet](https://camo.githubusercontent.com/d713741348fd88809ec0809de0a9aea7a6358b04f7d2aace673c9286ee290dfb/68747470733a2f2f7261772e6769746875622e636f6d2f6170616368652f706172717565742d666f726d61742f6d61737465722f646f632f696d616765732f46696c654c61796f75742e676966)

El formato Parquet incluye, además de los datos, el esquema de los mismos, lo que hace que se pueda leer sin dar lugar a errores. Esto soluciona el problema que nos encontramos en CSV y JSON, que no incluyen el esquema de los datos.

Es incluso fomentado por el Gobierno de España para publicación de los datos: https://datos.gob.es/es/blog/por-que-deberias-de-usar-ficheros-parquet-si-procesas-muchos-datos.

En Python se puede leer con la biblioteca `pyarrow` (https://arrow.apache.org/docs/python/parquet.html).


In [ ]:
%pip install pyarrow

In [ ]:
# Write the df dataframe to parquet file
import pandas as pd
import time

start_time: float = time.time()
df_csv: pd.DataFrame = pd.read_csv('Posts.csv')
print("Tiempo de lectura CSV: %s segundos" % (time.time() - start_time))
df_csv.to_parquet('Posts.parquet', compression='snappy')

In [ ]:
!ls -lh Posts.*

Se puede ver el contenido del fichero Parquet, pero no es legible, porque está en binario.

In [ ]:
!head Posts.parquet

In [ ]:
start_time: float = time.time()
df_parquet: pd.DataFrame = pd.read_parquet('Posts.parquet')
print("Tiempo de lectura Parquet: %s segundos" % (time.time() - start_time))

Téngase en cuenta que el Parquet guarda la información de tipos de datos, por lo que no es necesario especificarlos.

In [ ]:
df_parquet.dtypes

In [ ]:
df_parquet.head()

In [ ]:
print("Tamaño del archivo CSV:", df_csv.memory_usage(deep=True).sum(), "bytes")

In [ ]:
print("Tamaño del archivo Parquet:", df_parquet.memory_usage(deep=True).sum(), "bytes")

Una de las ventajas del formato Parquet es que puedes leer solo las columnas que necesitas, lo cual es útil para trabajar con grandes conjuntos de datos.

In [ ]:
df_parquet_subset: pd.DataFrame = pd.read_parquet('Posts.parquet', columns=['PostTypeId', 'CreationDate'])
print(df_parquet_subset.head())

Se pueden dividir los datos en varios archivos Parquet (por ejemplo, particionados por el tipo de licencia, score, etc.) y luego cargar esos archivos de manera eficiente.

In [39]:
df_parquet.to_parquet('Posts_partitioned/', partition_cols=['ContentLicense'])

In [ ]:
!ls -R Posts_partitioned

In [ ]:
!tree "Posts_partitioned"